# Módulo 07 · Generación con citas

Se construye el prompt con los fragmentos numerados, se genera la respuesta y se
vinculan las citas con sus fuentes.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
from fundamenta.chunking import chunk_documents
from fundamenta.generation import NO_ANSWER, SYSTEM_PROMPT, build_prompt, to_sources
from fundamenta.loaders import load_directory
from fundamenta.retriever import Retriever

chunks = chunk_documents(load_directory(DATA_DIR))
retriever = Retriever(chunks, mode="lexical")

question = "¿Cuántos días de vacaciones tengo y cómo se solicitan?"
sources = to_sources(retriever.search(question, k=3))
for source in sources:
    print(f"[{source.number}] {source.label}")

[1] manual-del-empleado.md › Vacaciones
[2] beneficios-2026.pdf › página 2
[3] manual-del-empleado.md › Permisos retribuidos


## Instrucciones del sistema

In [3]:
print(SYSTEM_PROMPT.format(organization="NovaTech", no_answer=NO_ANSWER))

Usted es el asistente de conocimiento interno de NovaTech.

Responda a la pregunta utilizando exclusivamente la información de los fragmentos de
documentación que se le proporcionan.

Normas:
- Cite cada afirmación con el número del fragmento entre corchetes, por ejemplo [1] o [2][3].
- No utilice conocimientos externos ni haga suposiciones sobre lo que no aparece en los fragmentos.
- Si los fragmentos no contienen la respuesta, responda exactamente: "No dispongo de información suficiente en la documentación disponible para responder a esta pregunta."
- Responda en el idioma de la pregunta, de forma clara y concisa.


## Prompt enviado al modelo

In [4]:
print(build_prompt(question, sources))

Fragmentos de documentación:

[1] Fuente: manual-del-empleado.md › Vacaciones
## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.

- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse
  antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5 días extra.

[2] Fuente: beneficios-2026.pdf › página 2
6. Día de voluntariado
Cada persona tiene 1 día laborable al año para colaborar con una ONG de su
elección. Este día no se descuenta de las vacaciones.
7. Programa de referidos
Si recomiendas a una persona y NovaTech la contrata, recibes 2.000 € brutos
cuando esa persona supera el periodo de prueba de 6 meses. El programa no aplica
a puestos de dirección ni a recomendaciones de familiares directos.
Para cualquie

## Conexión con un proveedor

El sistema admite cualquier proveedor que implemente `generate(system, prompt)`:

```python
from fundamenta.llm import ClaudeLLM, OllamaLLM

llm = ClaudeLLM()                      # Claude (requiere ANTHROPIC_API_KEY)
llm = OllamaLLM(model="qwen2.5:7b")    # modelo abierto en local, sin salida de datos
```

## Citas

La respuesta del modelo incluye referencias como `[1]`. El sistema conserva como
fuentes únicamente los fragmentos citados:

In [5]:
from fundamenta.generation import cited_numbers

example = "Dispone de 23 días laborables al año [1]. Se solicitan en NovaHR con 15 días de antelación [1]."
print("Fragmentos citados:", cited_numbers(example))

Fragmentos citados: {1}


## Modo extractivo

Sin LLM configurado, el sistema devuelve los fragmentos relevantes con sus
referencias. Permite demostraciones sin credenciales y sirve como alternativa si el
proveedor no está disponible.

In [6]:
from fundamenta.generation import AnswerGenerator

answer = AnswerGenerator().answer(question, retriever.search(question, k=2))
print(answer.mode)
print(answer.text[:600])

extractive
Fragmentos más relevantes de la documentación:

[1] manual-del-empleado.md › Vacaciones
## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.

- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse
  antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5 días extra.

[2] beneficios-2026.pdf › página 2
6. Día de vol
